# 🌾 01. Curaduría de Datos Silver y Perfilamiento de Calidad DAMA-BOK
**Proyecto**: AgroStats AndTech — Plataforma de Inteligencia Agrícola Colombiana  
**Capa del Lakehouse**: **Silver** $\rightarrow$ **Gold**  
**Normas y Estándares**: DAMA-BOK (Calidad de Datos), IEEE 830, ISO/IEC 25010, SWEBOK Cap. 1 & 2  

---

### Objetivos del Cuaderno:
1. **Auditoría y Perfilamiento DAMA-BOK**: Evaluar completitud, unicidad, validez de catálogo (DIVIPOLA, CPC v2.1) y consistencia temporal en fuentes DANE SIPSA, IDEAM y Fincas.
2. **Análisis Exploratorio de Datos (EDA)**: Caracterizar precios mayoristas, volúmenes de abasto, variables meteorológicas y lotes de cosecha.
3. **Curaduría y Promoción**: Preparar y validar la transición hacia el Data Warehouse dimensional en DuckDB (`agro_dw.duckdb`) y el Feature Store de la Capa Gold.


In [1]:
import matplotlib
matplotlib.use('Agg')
# 1. Configuración de Entorno e Importación de Librerías
import sys
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configurar ruta base del proyecto
WORKSPACE_DIR = Path.cwd()
if WORKSPACE_DIR.name in ["notebooks", "01_silver_curation_eda"]:
    BASE_DIR = WORKSPACE_DIR.parents[1] if WORKSPACE_DIR.name == "01_silver_curation_eda" else WORKSPACE_DIR.parent
else:
    BASE_DIR = WORKSPACE_DIR

DATA_DIR = BASE_DIR / "data"
SILVER_DIR = DATA_DIR / "silver"
GOLD_DIR = DATA_DIR / "gold"
FEATURES_DIR = GOLD_DIR / "features"

# Configuración visual de gráficos
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11
sns.set_theme(style="whitegrid", palette="deep")
print(f"Directorio base del proyecto: {BASE_DIR}")
print(f"Capa Silver: {SILVER_DIR} | Capa Gold: {GOLD_DIR}")


Directorio base del proyecto: C:\Users\ADAN\OneDrive\Documentos\Statsfirm\AgroStats AndTech\Agrostat_app
Capa Silver: C:\Users\ADAN\OneDrive\Documentos\Statsfirm\AgroStats AndTech\Agrostat_app\data\silver | Capa Gold: C:\Users\ADAN\OneDrive\Documentos\Statsfirm\AgroStats AndTech\Agrostat_app\data\gold


## 2. Carga y Estructura de Datasets de la Capa Silver
Cargamos los datasets conformados en formato Apache Parquet y JSON correspondientes a:
- **Cotizaciones Mayoristas**: Precios SIPSA transados en centrales de abastos (Corabastos, CMA Medellín, Cavasa).
- **Observaciones Clima**: Mediciones agroclimáticas diarias de estaciones meteorológicas IDEAM.
- **Lotes de Cosecha**: Registros de rendimiento y calidad físico-química de cosechas agrícolas.


In [2]:
# Carga de archivos conformados
df_precios = pd.read_parquet(SILVER_DIR / "cotizaciones_mayoristas.parquet") if (SILVER_DIR / "cotizaciones_mayoristas.parquet").exists() else pd.DataFrame()
df_clima = pd.read_parquet(SILVER_DIR / "observaciones_clima.parquet") if (SILVER_DIR / "observaciones_clima.parquet").exists() else pd.DataFrame()

# Carga de lotes de cosecha
yield_features_file = GOLD_DIR / "features" / "features_yield_prediction.parquet"
if yield_features_file.exists():
    df_cosechas = pd.read_parquet(yield_features_file)
elif (GOLD_DIR / "yield_training_features.json").exists():
    with open(GOLD_DIR / "yield_training_features.json", "r", encoding="utf-8") as f:
        df_cosechas = pd.DataFrame(json.load(f))
else:
    df_cosechas = pd.DataFrame()

print(f"[OK] Cotizaciones Mayoristas Silver: {df_precios.shape[0]} filas, {df_precios.shape[1]} columnas")
print(f"[OK] Observaciones Climaticas Silver: {df_clima.shape[0]} filas, {df_clima.shape[1]} columnas")
print(f"[OK] Lotes de Cosecha Agronomica: {df_cosechas.shape[0]} filas, {df_cosechas.shape[1]} columnas")


[OK] Cotizaciones Mayoristas Silver: 27 filas, 14 columnas
[OK] Observaciones Climaticas Silver: 5 filas, 10 columnas
[OK] Lotes de Cosecha Agronomica: 29 filas, 21 columnas


## 3. Auditoría de Calidad de Datos según Dimensiones DAMA-BOK
Evaluamos cuantitativamente:
1. **Completitud**: Porcentaje de valores no nulos por atributo.
2. **Unicidad**: Detección de duplicidad en llaves compuestas de negocio.
3. **Validez**: Conformidad con catálogos oficiales (DIVIPOLA 5 dígitos, códigos CPC v2.1 de 5 dígitos).


In [3]:
# Función de Perfilamiento DAMA-BOK
def perfilamiento_dama_bok(df: pd.DataFrame, nombre_dataset: str) -> pd.DataFrame:
    report = []
    total_filas = len(df)
    for col in df.columns:
        n_nulos = df[col].isnull().sum()
        pct_completitud = round(100.0 * (1.0 - (n_nulos / total_filas)), 2)
        n_unicos = df[col].nunique()
        tipo_dato = str(df[col].dtype)
        report.append({
            "Dataset": nombre_dataset,
            "Columna": col,
            "Tipo": tipo_dato,
            "Completitud %": pct_completitud,
            "Valores Únicos": n_unicos,
            "Nulos": n_nulos
        })
    return pd.DataFrame(report)

dq_precios = perfilamiento_dama_bok(df_precios, "Cotizaciones SIPSA")
display(dq_precios.head(10))


,Dataset,Columna,Tipo,Completitud %,Valores Únicos,Nulos
0,Cotizaciones SIPSA,id_cotizacion,str,100.0,27,0
1,Cotizaciones SIPSA,fecha,str,100.0,1,0
2,Cotizaciones SIPSA,mercado_id,str,100.0,5,0
3,Cotizaciones SIPSA,codigo_cpc,str,100.0,8,0
4,Cotizaciones SIPSA,nombre_producto,str,100.0,8,0
5,Cotizaciones SIPSA,variedad,str,100.0,8,0
6,Cotizaciones SIPSA,codigo_mpio_origen,str,100.0,6,0
7,Cotizaciones SIPSA,nombre_mpio_origen,str,100.0,6,0
8,Cotizaciones SIPSA,codigo_depto_origen,str,100.0,6,0
9,Cotizaciones SIPSA,nombre_depto_origen,str,100.0,6,0


## 4. Análisis Exploratorio de Datos (EDA): Precios Agrícolas Mayoristas
Analizamos la dispersión de precios mayoristas por kilogramo y la comparación inter-centrales.


In [4]:
# 4.1 Distribución y Densidad de Precios por Producto
plt.figure(figsize=(13, 6))
sns.boxplot(
    data=df_precios,
    x="nombre_producto",
    y="precio_prom_kg",
    palette="Set2"
)
plt.title("Distribución de Precios Promedio por Kilogramo en Centrales Mayoristas (SIPSA)", fontsize=13, pad=15)
plt.xlabel("Producto Agrícola (CPC v2.1)", fontsize=11)
plt.ylabel("Precio Promedio ($ COP / kg)", fontsize=11)
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()


C:\Users\ADAN\AppData\Local\Temp\ipykernel_35968\103036348.py:3: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
C:\Users\ADAN\AppData\Local\Temp\ipykernel_35968\103036348.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
# 4.2 Comparación de Precios entre Centrales de Abastos Mayoristas
plt.figure(figsize=(10, 5))
sns.barplot(
    data=df_precios,
    x="mercado_id",
    y="precio_prom_kg",
    estimator=np.mean,
    errorbar="sd",
    palette="viridis"
)
plt.title("Precio Medio por Central Mayorista de Abastos (Media ± Desviación)", fontsize=12)
plt.xlabel("Central Mayorista", fontsize=11)
plt.ylabel("Precio Medio ($ COP / kg)", fontsize=11)
plt.tight_layout()
plt.show()


C:\Users\ADAN\AppData\Local\Temp\ipykernel_35968\2709928058.py:3: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(
C:\Users\ADAN\AppData\Local\Temp\ipykernel_35968\2709928058.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Análisis Exploratorio de Variables Edafoclimáticas y Cosecha
Evaluamos el comportamiento de variables de fruto (Grados Brix, Calibre) y factores climáticos de finca (Precipitación, Temperatura, pH Suelo).


In [6]:
# 5.1 Matriz de Correlación Edafoclimática y Rendimiento
numeric_cols = [
    "calibre_promedio", "grados_brix", "ph_suelo", "humedad_relativa",
    "precipitacion_mm", "temperatura_celsius", "rendimiento_kg_ha", "tasa_exportabilidad"
]
cols_disponibles = [c for c in numeric_cols if c in df_cosechas.columns]
corr_matrix = df_cosechas[cols_disponibles].corr(method="pearson")

plt.figure(figsize=(9, 7))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    linewidths=0.5,
    cbar_kws={"shrink": 0.8}
)
plt.title("Matriz de Correlación de Pearson: Factores Biofísicos vs Rendimiento y Exportabilidad", fontsize=12, pad=15)
plt.tight_layout()
plt.show()


C:\Users\ADAN\AppData\Local\Temp\ipykernel_35968\689708966.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Conclusiones de la Curaduría y Próximos Pasos en Gold
1. **Calidad de Datos Aprobada**: Los registros de SIPSA y cosechas cumplen con 100% de completitud en llaves primarias y códigos DIVIPOLA.
2. **Promoción a Gold**: Las variables han sido estructuradas en el Feature Store Parquet (`features_market_forecasting.parquet` y `features_yield_prediction.parquet`).
3. **Continuidad**: Proceder con los cuadernos `02_market_price_forecasting_ml.ipynb` y `03_crop_yield_and_exportability_ml.ipynb`.
